# 作业一：Data Types, Functions and Conditionals

姓名：谭嘉舜
学号：20244080208

## 身份与环境记录

In [2]:
import os, uuid, hashlib, platform, getpass
# ===== 1. 填写学号 =====
STUDENT_ID = 20190010 #修改为自己的学号
# ===== 2. 打印 conda 环境 =====
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")
# ===== 3. 机器指纹（绑定当前电脑）=====
MACHINE_FP = hashlib.sha256(
 f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
# ===== 4. 生成唯一 ID =====
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")

[Conda 环境] course-env
[学号] 20190010
[机器指纹] 2d0321b280a8536f
[RUN_ID] 20190010-2d0321b280a8536f


## 一、热身：定义简单函数

分别定义无参数函数、字符串前后添加"goat"的函数，以及按指定次数逐行打印字符串的函数。

In [3]:
def say_hi():
    print("Hello, world!")


def goat_pad(s):
    print("goat" + s + "goat")


def print_n(s, n):
    for i in range(n):
        print(s)


# 测试
say_hi()
goat_pad("Python")
print_n("学习Python", 3)

Hello, world!
goatPythongoat
学习Python
学习Python
学习Python


## 二、欧几里得算法

利用辗转相除法计算最大公约数：反复用较小数和余数替换原来的两个数，直到余数为0。

In [4]:
def gcd(m, n):
    # 将负数转换为绝对值
    m = abs(m)
    n = abs(n)

    while n != 0:
        m, n = n, m % n

    return m


# 题目要求的三组测试
for m, n in [(2018, 2019), (1200, 300), (5040, 60)]:
    print(f"gcd({m}, {n}) = {gcd(m, n)}")

# 负数测试
for m, n in [(-1200, 300), (1200, -300), (-1200, -300)]:
    print(f"gcd({m}, {n}) = {gcd(m, n)}")

gcd(2018, 2019) = 1
gcd(1200, 300) = 300
gcd(5040, 60) = 60
gcd(-1200, 300) = 300
gcd(1200, -300) = 300
gcd(-1200, -300) = 300


本函数先取两个输入的绝对值，因此一个或两个参数为负数时，返回对应绝对值的非负最大公约数。
这种行为有意义，因为改变整数的正负号不影响它们的正公约数。

## 三、近似欧拉数e

### 1. 使用极限公式
使用(1 + 1/n)的n次方近似e，以下测试使用正整数n。

In [5]:
import math


def euler_limit(n):
    return float((1 + 1 / n) ** n)


for n in [1, 10, 100, 1000, 10000]:
    print(f"n={n}，近似值={euler_limit(n):.10f}")

print(f"e的参考值：{math.exp(1):.10f}")

n=1，近似值=2.0000000000
n=10，近似值=2.5937424601
n=100，近似值=2.7048138294
n=1000，近似值=2.7169239322
n=10000，近似值=2.7181459268
e的参考值：2.7182818285


### 2. 使用级数前n项求和
从k=0开始累加1/k!，共计算n项；n=0时返回0.0。

In [6]:
def euler_infinite_sum(n):
    total = 0.0
    factorial = 1

    for k in range(n):
        if k > 0:
            factorial *= k

        total += 1 / factorial

    return total


for n in [0, 1, 4, 10]:
    result = euler_infinite_sum(n)
    print(f"n={n}，近似值={result}，类型={type(result)}")

n=0，近似值=0.0，类型=<class 'float'>
n=1，近似值=1.0，类型=<class 'float'>
n=4，近似值=2.6666666666666665，类型=<class 'float'>
n=10，近似值=2.7182815255731922，类型=<class 'float'>


前n项对应k=0到n−1，而不是k=0到n。
当n=4时，计算1/0! + 1/1! + 1/2! + 1/3!，结果约为2.667。
total初始化为0.0，因此无参数项可累加时也返回浮点数。

### 3. 按误差要求累加级数
使用while循环累加，直到近似值与math.exp(1)的绝对差不超过epsilon。
epsilon应为正数。

In [7]:
def euler_approx(epsilon):
    if epsilon <= 0:
        raise ValueError("epsilon必须大于0")

    true_e = math.exp(1)
    total = 0.0
    term = 1.0
    k = 0

    while abs(true_e - total) > epsilon:
        previous_total = total
        total += term

        # 防止精度要求过高时循环无法结束
        if total == previous_total:
            raise ValueError("误差要求超出了当前浮点计算能达到的精度")

        k += 1
        term /= k

    return total


for epsilon in [0.1, 0.001, 0.000001]:
    approximation = euler_approx(epsilon)
    error = abs(math.exp(1) - approximation)

    print(
        f"epsilon={epsilon}，"
        f"近似值={approximation:.10f}，"
        f"绝对误差={error:.10f}，"
        f"满足要求={error <= epsilon}"
    )

epsilon=0.1，近似值=2.6666666667，绝对误差=0.0516151618，满足要求=True
epsilon=0.001，近似值=2.7180555556，绝对误差=0.0002262729，满足要求=True
epsilon=1e-06，近似值=2.7182815256，绝对误差=0.0000003029，满足要求=True


term从1/0!开始，每次通过除以下一个整数得到下一项。
while循环检查实际绝对误差，满足要求后返回近似值。
math.exp(1)作为浮点精度下的参考值；浮点计算无法保证任意小的误差。

### 4. 逐行打印级数求和结果
依次打印euler_infinite_sum(k)，其中k从1到n。

In [8]:
def print_euler_sum_table(n):
    for k in range(1, n + 1):
        print(euler_infinite_sum(k))


print_euler_sum_table(6)

1.0
2.0
2.5
2.6666666666666665
2.708333333333333
2.7166666666666663


## 四、测试整数属性

### 1. 判断正整数是否为2的幂
反复检查能否被2整除，并进行整数除法；最后为1则是2的幂。

In [9]:
def is_power_of_2(n):
    while n > 1:
        if n % 2 != 0:
            return False

        n //= 2

    return True


for n in [1, 2, 3, 4, 8, 12, 16, 32]:
    print(f"{n}是否为2的幂：{is_power_of_2(n)}")

1是否为2的幂：True
2是否为2的幂：True
3是否为2的幂：False
4是否为2的幂：True
8是否为2的幂：True
12是否为2的幂：False
16是否为2的幂：True
32是否为2的幂：True


1是2的0次方，因此返回True。
本函数按照题目假设接收正整数，使用取余和整数除法进行判断，没有使用平方根或对数。

### 2. 判断正整数n是否为正整数b的幂
对于b大于1的情况，反复整除；底数为1时，只有n=1满足条件。

In [10]:
def is_power(b, n):
    if b == 1:
        return n == 1

    while n > 1:
        if n % b != 0:
            return False

        n //= b

    return True


test_pairs = [
    (2, 16),
    (2, 12),
    (3, 27),
    (3, 12),
    (5, 125),
    (5, 1),
    (1, 1),
    (1, 5),
]

for b, n in test_pairs:
    print(f"is_power({b}, {n}) = {is_power(b, n)}")

is_power(2, 16) = True
is_power(2, 12) = False
is_power(3, 27) = True
is_power(3, 12) = False
is_power(5, 125) = True
is_power(5, 1) = True
is_power(1, 1) = True
is_power(1, 5) = False


任意正整数b的0次方均为1，所以n=1时返回True。
当b=1时，它的任何非负整数次方都为1，因此只有n=1满足条件。
当b大于1时，如果某一步不能整除，就不是b的幂；如果最终除到1，则是b的幂。